# Class 7 - NumPy Fundamentals
### Data Analytics in Python | Week 3, Friday

Python lists are flexible but slow. When you're computing on a million patient
readings, a million gene expression values, or a million simulated outcomes, that
slowness becomes a blocker.

NumPy solves this by storing numbers in contiguous memory blocks and executing
operations in compiled C code - with no Python overhead per element. The result
is typically **20–100× faster** than equivalent Python loops. Every scientific
Python library - Pandas, Matplotlib, Scikit-learn, Scipy - is built on NumPy arrays
underneath. Learning NumPy is not optional; it is the foundation.

**Today:**
- Why NumPy: the memory model that makes it fast
- Array creation: seven methods you'll use constantly
- Array attributes: shape, dtype, ndim, size, nbytes
- Indexing and slicing: 1D and 2D
- Views vs copies - a trap that causes real bugs
- Data types and why they matter in analytics
- Practical: Array slicing lab

## 1. Why NumPy - the speed argument

Before any syntax: why bother? Let's measure first, explain second.

In [3]:
import numpy as np
import time

N = 1_000_000

# Python list: square each element
py_list = list(range(N))
t0 = time.time()
py_result = [x ** 2 for x in py_list]
py_time = time.time() - t0

# NumPy array: same operation
np_array = np.arange(N)
t0 = time.time()
np_result = np_array ** 2
np_time = time.time() - t0

print(f"Task: square {N:,} numbers")
print(f"  Python list + comprehension: {py_time*1000:.2f} ms")
print(f"  NumPy array:                 {np_time*1000:.2f} ms")
print(f"  NumPy is {py_time / (np_time + 1e-10):.0f}x faster")
print()
print(f"Memory: Python list holds {N:,} Python int objects")
print(f"  Each Python int: ~28 bytes")
print(f"  Total list memory: ~{N * 28 / 1e6:.0f} MB")
print(f"  NumPy int64 array: {np_array.nbytes / 1e6:.1f} MB  ({np_array.itemsize} bytes/element)")
print(f"  Memory saving: {N*28 / np_array.nbytes:.1f}x")

Task: square 1,000,000 numbers
  Python list + comprehension: 61.86 ms
  NumPy array:                 1.51 ms
  NumPy is 41x faster

Memory: Python list holds 1,000,000 Python int objects
  Each Python int: ~28 bytes
  Total list memory: ~28 MB
  NumPy int64 array: 4.0 MB  (4 bytes/element)
  Memory saving: 7.0x


In [6]:
%%time      
#tells the time required to execute the full program

# %time       # time required to execute a single line

py_result = [x ** 2 for x in py_list]

CPU times: total: 78.1 ms
Wall time: 84.4 ms


In [8]:
%%time
np_result = np_array ** 2

CPU times: total: 0 ns
Wall time: 3 ms


**What makes NumPy fast:**
- Arrays store raw numbers in a *contiguous* block - no Python object overhead per element
- Operations execute in compiled C, processing the entire array without a Python loop
- Modern CPUs can apply one instruction to multiple elements at once (SIMD/vectorization)

A Python list of a million integers takes ~28 MB. The same data as a NumPy `int64` array takes 8 MB. And NumPy can compute on it 20–100× faster.

## 2. Array creation - seven methods

### 1. From a Python list or nested list

In [ ]:
a1d = np.array([1, 2, 3, 4, 5])
a2d = np.array([[1, 2, 3],
                [4, 5, 6],
                [7, 8, 9]])
print(f"From list (1D): {a1d}")
print(f"From list (2D):\n{a2d}")

From list (1D): [1 2 3 4 5]
From list (2D):
[[1 2 3]
 [4 5 6]
 [7 8 9]]


### 2. np.zeros and np.ones - all zeros or all ones

In [ ]:
z = np.zeros((3, 4))          # 3 rows, 4 cols, float64 by default
o = np.ones((2, 3), dtype=np.int32)
f = np.full((2, 3), 7.5)      # fill with a constant

print(f"zeros (3,4):\n{z}")
print(f"ones  (2,3) int32:\n{o}")
print(f"full  (2,3) = 7.5:\n{f}")

zeros (3,4):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
ones  (2,3) int32:
[[1 1 1]
 [1 1 1]]
full  (2,3) = 7.5:
[[7.5 7.5 7.5]
 [7.5 7.5 7.5]]


### 3. np.arange - like range(), but returns a NumPy array

In [ ]:
a = np.arange(0, 10)          # integers 0..9
b = np.arange(0, 1, 0.1)      # floats 0.0, 0.1, ..., 0.9
c = np.arange(10, 0, -2)      # descending

print(f"arange(0,10):     {a}")
print(f"arange(0,1,0.1):  {b.round(1)}")
print(f"arange(10,0,-2):  {c}")

arange(0,10):     [0 1 2 3 4 5 6 7 8 9]
arange(0,1,0.1):  [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
arange(10,0,-2):  [10  8  6  4  2]

linspace(36,40,9): [36.  36.5 37.  37.5 38.  38.5 39.  39.5 40. ]


### 4. np.linspace - N evenly spaced values BETWEEN start and end (inclusive)

In [1]:
temps = np.linspace(36.0, 40.0, 9)   # 9 values from 36.0 to 40.0
print(f"\nlinspace(36,40,9): {temps}")

<IPython.core.display.Javascript object>


linspace(36,40,9): [36.  36.5 37.  37.5 38.  38.5 39.  39.5 40. ]


### 5. np.eye - identity matrix

In [2]:
I = np.eye(4)
print(f"Identity (4x4):\n{I}")

<IPython.core.display.Javascript object>

Identity (4x4):
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


### 6. np.random - random arrays (Chapter 9 covers this in depth)

In [3]:
np.random.seed(42)
r = np.random.randint(0, 100, size=(3, 4))
print(f"\nRandom int (3,4):\n{r}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Random int (3,4):
[[51 92 14 71]
 [60 20 82 86]
 [74 74 87 99]]


### 7. Conversion from existing data

In [4]:
python_list = [[1.1, 2.2], [3.3, 4.4]]
arr = np.asarray(python_list)          # zero-copy when possible
print(f"\nFrom nested list:\n{arr}")

<IPython.core.display.Javascript object>


From nested list:
[[1.1 2.2]
 [3.3 4.4]]


### 8. Initializing Random Matrices for ML
When building machine learning models (like initializing weights in a neural network), we rarely start with zeros or ones. Instead, we fill matrices with random numbers drawn from specific statistical distributions.

- `np.random.randn(rows, cols)` --> Creates an array filled with random floats from a **Standard Normal Distribution** (Mean = 0, Standard Deviation = 1).
- `np.random.rand(rows, cols)` --> Creates an array filled with random floats uniformly distributed between `0` and `1`.

In [5]:
#simulate initializing a weight matrix for a neural network layer
# 4 inputs connecting to 3 hidden neurons
np.random.seed(42)
weights_normal = np.random.randn(4, 3) 
print("Weights from a Standard Normal Distribution (centered around 0):")
print(weights_normal.round(4))

#generate random uniform probabilities or scaling factors
probabilities = np.random.rand(2, 5)
print("\nRandom factors strictly uniform between [0, 1):")
print(probabilities.round(4))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Weights from a Standard Normal Distribution (centered around 0):
[[ 0.4967 -0.1383  0.6477]
 [ 1.523  -0.2342 -0.2341]
 [ 1.5792  0.7674 -0.4695]
 [ 0.5426 -0.4634 -0.4657]]


<IPython.core.display.Javascript object>


Random factors strictly uniform between [0, 1):
[[0.3042 0.5248 0.4319 0.2912 0.6119]
 [0.1395 0.2921 0.3664 0.4561 0.7852]]


## 3. Array attributes

Four attributes you need every single time you work with a new array: know the shape before you index, the dtype before you compute, the size before you loop, the memory before you allocate.

In [6]:
# Build a gene expression matrix: 4 patients x 6 genes
gene_expr = np.array([
    [1.23, 0.87, 2.14, 1.05, 0.76, 1.89],
    [0.91, 1.44, 0.78, 3.21, 1.12, 0.55],
    [2.05, 1.89, 1.33, 0.64, 2.41, 1.77],
    [1.56, 0.72, 1.95, 1.38, 0.93, 2.12],
])

print(f"shape    : {gene_expr.shape}")     # (rows, cols)
print(f"ndim     : {gene_expr.ndim}")      # number of dimensions
print(f"size     : {gene_expr.size}")      # total elements
print(f"dtype    : {gene_expr.dtype}")     # data type
print(f"itemsize : {gene_expr.itemsize}")  # bytes per element
print(f"nbytes   : {gene_expr.nbytes}")    # total bytes

# These are the first four things you check with a new dataset
# shape tells you n_samples x n_features
# dtype tells you whether you'll have precision issues
# nbytes tells you whether it fits in RAM

shape    : (4, 6)
ndim     : 2
size     : 24
dtype    : float64
itemsize : 8
nbytes   : 192


### 3a. Data types - choosing the right dtype saves memory and prevents bugs

In [ ]:
dtypes = [np.int8, np.int16, np.int32, np.int64,
          np.float32, np.float64, np.bool_]

print(f"{'dtype':<12} {'bytes':>8} {'range/precision'}")
print("-" * 50)
info = {
    np.int8:    "[-128, 127]",
    np.int16:   "[-32768, 32767]",
    np.int32:   "[-2.1B, 2.1B]",
    np.int64:   "[-9.2e18, 9.2e18]",
    np.float32: "~7 decimal places",
    np.float64: "~15 decimal places",
    np.bool_:   "True/False",
}
for dt in dtypes:
    arr = np.array([1], dtype=dt)
    print(f"  {np.dtype(dt).name:<12} {arr.itemsize:>6}B   {info[dt]}")

# Practical: for 1M patient scores (0-100), use uint8 (1 byte) not float64 (8 bytes)
scores_f64 = np.random.randint(0, 101, 1_000_000).astype(np.float64)
scores_u8  = scores_f64.astype(np.uint8)
print(f"\n1M scores as float64: {scores_f64.nbytes/1e6:.1f} MB")
print(f"1M scores as uint8:   {scores_u8.nbytes/1e6:.1f} MB")

dtype           bytes range/precision
--------------------------------------------------
  int8              1B   [-128, 127]
  int16             2B   [-32768, 32767]
  int32             4B   [-2.1B, 2.1B]
  int64             8B   [-9.2e18, 9.2e18]
  float32           4B   ~7 decimal places
  float64           8B   ~15 decimal places
  bool              1B   True/False

1M scores as float64: 8.0 MB
1M scores as uint8:   1.0 MB


### **3a. Data Types, Memory Layouts, and Numerical Precision**

In NumPy, selecting an explicit data type (`dtype`) represents a fundamental balance between **numerical scale** and **memory overhead**. Unlike native Python dynamic typing, NumPy arrays rely on homogeneous, fixed-size memory blocks. Failing to specify optimal data types can lead to memory exhaustion when processing large-scale datasets, whereas choosing overly restrictive types introduces numerical overflow.

### 3.1 Memory Calculations: Bits and Bytes

* A **bit** is the fundamental unit of digital storage, representing either $0$ or $1$.
* A **byte** consists of **8 bits**.

When a NumPy data type has a number in its name (like `int8`, `int16`, `int32`), that number tells you exactly how many **bits** it uses. To find the bytes, we just divide by 8:

* $\text{int8}  \implies 8 \text{ bits} \div 8 = \mathbf{1 \text{ Byte}}$
* $\text{int16} \implies 16 \text{ bits} \div 8 = \mathbf{2 \text{ Bytes}}$
* $\text{int32} \implies 32 \text{ bits} \div 8 = \mathbf{4 \text{ Bytes}}$
* $\text{int64} \implies 64 \text{ bits} \div 8 = \mathbf{8 \text{ Bytes}}$


### 3.2 Calculating Numerical Ranges Across Data Types

The total number of unique binary states an $n$-bit container can represent is defined by:

$$\text{Total Unique Values} = 2^{n}$$

#### Unsigned Integers (`uint`)
Unsigned integer data types store non-negative values exclusively. Because no bits are allocated for sign tracking, the range begins at $0$:

$$\text{Range}_{\text{unsigned}} = \left[0,\, 2^{n} - 1\right]$$

* **Example (`uint8`):** $2^8 = 256$ total states $\implies$ **Range: $0$ to $255$**.

#### Signed Integers (`int`)
To store Signed integer data types (negative numbers), the computer splits the total combinations perfectly in half: one half for negative numbers, and the other half for zero and positive numbers.:

$$\text{Range}_{\text{signed}} = \left[-2^{n-1},\, 2^{n-1} - 1\right]$$

* **Example (`int8`):** $2^8 = 256$ total states, yielding $128$ negative values and $128$ non-negative values (including $0$) $\implies$ **Range: $-128$ to $127$**.


### 3.3 Summary of Essential NumPy Data Types

| Data Type | Memory Footprint | Theoretical Boundary / Precision | Numerical Range / Domain | Recommended Use Cases |
| :--- | :--- | :--- | :--- | :--- |
| **`int8`** | 1 Byte (8 bits) | $-2^7 \text{ to } 2^7 - 1$ | $-128 \text{ to } 127$ | Low-range categorical features, compact ordinal scores. |
| **`uint8`** | 1 Byte (8 bits) | $0 \text{ to } 2^8 - 1$ | $0 \text{ to } 255$ | Image pixel intensities (RGB), non-negative counts. |
| **`int16`** | 2 Bytes (16 bits) | $-2^{15} \text{ to } 2^{15} - 1$ | $-32,768 \text{ to } 32,767$ | Intermediate integer ranges, audio wave signal data. |
| **`uint16`** | 2 Bytes (16 bits) | $0 \text{ to } 2^{16} - 1$ | $0 \text{ to } 65,535$ | High-depth medical imaging, medium scale frequencies. |
| **`int32`** | 4 Bytes (32 bits) | $-2^{31} \text{ to } 2^{31} - 1$ | $\approx -2.14 \times 10^9 \text{ to } 2.14 \times 10^9$ | General-purpose array indexing, discrete counting. |
| **`int64`** | 8 Bytes (64 bits) | $-2^{63} \text{ to } 2^{63} - 1$ | $\approx -9.22 \times 10^{18} \text{ to } 9.22 \times 10^{18}$ | High-density genomic base-pair coordinates, global transaction IDs. |
| **`float32`** | 4 Bytes (32 bits) | $\sim 7$ significant decimal digits | IEEE 754 single-precision floating-point | Deep learning model weights, memory-constrained real numbers. |
| **`float64`** | 8 Bytes (64 bits) | $\sim 15$ significant decimal digits | IEEE 754 double-precision floating-point | Scientific computing, high-precision mathematical models (NumPy default). |
| **`bool_`** | 1 Byte (8 bits) | Single bit storage (byte-addressed) | `True` or `False` | Boolean masking, logical evaluation arrays. |

> **Memory Addressing Note on `bool_`:** Although a boolean logically requires only a single bit to represent True or False, modern computing architectures address memory at byte boundaries. Consequently, every standard `bool_` element allocates $1$ full byte ($8$ bits) in RAM.


### 3.4 Theoretical vs. Efficient Allocation: Practical Examples

To understand the practical impact of explicit type selection, consider how memory allocation scales across different data domains.

#### Example 1: Clinical Survey Metrics
Suppose we store $1,000,000$ patient clinical survey scores restricted to integer values in the range $[0, 100]$. Think of computer's RAM as a warehouse full of storage boxes. A data type (`dtype`) determines the physical size of the box we allocate for **every single one** of our 1 million numbers.

1. **Default Allocation (`float64`):**
  - **Size:** 64 bits = 8 bytes of computer memory per box.
  - **Capacity:** It can hold massive decimals up to 15 digits of precision.
  
     $$1,000,000 \text{ elements} \times 8 \text{ bytes} = 8,000,000 \text{ bytes} \approx \mathbf{8.0 \text{ MB}}$$

  - **The Problem:** Using a massive, high-precision decimal box to store a simple whole number like `85` is total overkill. You are paying for space you aren't using.

2. **Optimal Allocation (`uint8`):**
   
   `uint8` stands for **U**nsigned (positive only) **Int**eger made of **8** bits.

  - **Size:** 8 bits = 1 byte of memory per box.
  - **Capacity:** Because an 8-bit box can only hold binary combinations from `00000000` to `11111111`, it can store any whole number from **0 to 255**.
  
     $$1,000,000 \text{ elements} \times 1 \text{ byte} = 1,000,000 \text{ bytes} \approx \mathbf{1.0 \text{ MB}}$$

  - **Why it works:** Since our patient scores are strictly between 0 and 100, every single possible score fits perfectly inside a box that goes up to 255. We just saved 7 MB of RAM by choosing a smaller box size. The integer's upper bound ($100$) fits comfortably within the bounds of `uint8` ($[0, 255]$):


By explicitly specifying `dtype=np.uint8`, we achieve an **8x reduction in RAM consumption** without introducing any truncation or loss of precision. When working with massive datasets, this is the difference between our code running smoothly or crashing our computer out of memory.


#### Example 2: Large-Scale Genomic Variant Matrix
Consider a computational genomics pipeline analyzing single-nucleotide polymorphism (SNP) data across $10,000$ bacterial genomes (Samples/rows) and $50,000$ genetic loci (Features/columns). The dataset comprises $500,000,000$ (500 Million data points) discrete variant calls encoded as `0` (wild-type), `1` (heterozygous mutation), or `2` (homozygous mutation).
* **Unoptimized Implementation (Default `float64`):**
  $$500,000,000 \text{ elements} \times 8 \text{ bytes} = 4,000,000,000 \text{ bytes} = \mathbf{4.0 \text{ GB}}$$
  *Consequence:* Unnecessary RAM pressure that impairs vectorized vector execution speed and limits local pipeline scalability. Uses massive amounts of RAM, and might crash an ordinary laptop.
* **Optimized Implementation (`uint8`):** You recognize the numbers `0, 1, 2` fit perfectly inside a 1-byte box, so you explicitly set `dtype=np.uint8`.
  $$500,000,000 \text{ elements} \times 1 \text{ byte} = 500,000,000 \text{ bytes} = \mathbf{0.5 \text{ GB (500 MB)}}$$
  *Consequence:* An 87.5% memory reduction that improves cache locality and accelerates matrix operations across downstream statistical models. Script will now load instantly, fits cleanly into memory, and computes matrix operations exponentially faster.

### 3.5 Decision Architecture for Data Type Selection

When defining numerical data pipelines, data type allocation follows a systematic evaluation process:

```python
                  What kind of data is it?
                             │
            ┌────────────────┴────────────────┐
      Whole Numbers (Integer Data)         Decimals
            │                                 │
   What is the MAX value?             How much precision?
   0 to 255    --> uint8              ~7 decimals  --> float32
   ±32,000     --> int16              ~15 decimals --> float64 (Default)
   ±2 Billion  --> int32
   Massive/Precise     --> int64
```

### 3b. Casting Data Types with `.astype()`
When loading raw arrays, NumPy will guess the data type. If it guesses incorrectly (e.g., imports integers as `float64`), you can change it explicitly using `.astype()`. This creates a brand new array with the modified memory layout.

In [6]:
#imagine parsing raw categorical pixel values loaded as floats
raw_pixels = np.array([255.0, 0.0, 128.0, 64.0])
print(f"Original array type: {raw_pixels.dtype}")

#cast down to unsigned 8-bit integers to drastically reduce memory footprints
clean_pixels = raw_pixels.astype(np.uint8)
print(f"Casted array type:   {clean_pixels.dtype}")
print(f"Clean integer values: {clean_pixels}")

<IPython.core.display.Javascript object>

Original array type: float64


<IPython.core.display.Javascript object>

Casted array type:   uint8
Clean integer values: [255   0 128  64]


## 4. Indexing and slicing

The syntax is the same as Python lists, extended to multiple dimensions. The most important rule: **stop is always exclusive**.

### 1D indexing - identical to lists

In [ ]:
a = np.array([10, 20, 30, 40, 50, 60, 70])

print("=== 1D ===")
print(f"a[0]    = {a[0]}")      # 10
print(f"a[-1]   = {a[-1]}")     # 70
print(f"a[2:5]  = {a[2:5]}")    # [30, 40, 50]
print(f"a[:3]   = {a[:3]}")     # [10, 20, 30]
print(f"a[4:]   = {a[4:]}")     # [50, 60, 70]
print(f"a[::2]  = {a[::2]}")    # [10, 30, 50, 70]
print(f"a[::-1] = {a[::-1]}")   # [70, 60, 50, 40, 30, 20, 10]

=== 1D ===
a[0]    = 10
a[-1]   = 70
a[2:5]  = [30 40 50]
a[:3]   = [10 20 30]
a[4:]   = [50 60 70]
a[::2]  = [10 30 50 70]
a[::-1] = [70 60 50 40 30 20 10]


### 2D indexing - row, col separated by comma

In [ ]:
m = np.array([
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12],
])

print("Matrix m:")
print(m)
print()

# Single element
print(f"m[0, 0]   = {m[0, 0]}")    # 1
print(f"m[1, 2]   = {m[1, 2]}")    # 7
print(f"m[-1, -1] = {m[-1, -1]}")  # 12

# Row and column slices
print(f"\nRow 0:       {m[0, :]}")     # [1, 2, 3, 4]
print(f"Row 1:       {m[1, :]}")     # [5, 6, 7, 8]
print(f"Column 1:    {m[:, 1]}")     # [2, 6, 10]
print(f"Column -1:   {m[:, -1]}")    # [4, 8, 12]

# Submatrix - top-right 2x2
print(f"\nTop-right 2x2:\n{m[0:2, 2:4]}")

# Every other row
print(f"\nEvery other row:\n{m[::2, :]}")

Matrix m:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

m[0, 0]   = 1
m[1, 2]   = 7
m[-1, -1] = 12

Row 0:       [1 2 3 4]
Row 1:       [5 6 7 8]
Column 1:    [ 2  6 10]
Column -1:   [ 4  8 12]

Top-right 2x2:
[[3 4]
 [7 8]]

Every other row:
[[ 1  2  3  4]
 [ 9 10 11 12]]


### Practical 2D indexing - clinical lab matrix

In [11]:
# Rows = patients (P1-P5), Cols = markers (WBC, RBC, Hb, Plt, Glucose)
markers = ["WBC", "RBC", "Hb", "Plt", "Glucose"]
labs = np.array([
    [8.2,  4.9, 13.5, 250, 5.1],   # Patient 1 - normal
    [12.1, 4.2, 11.8, 380, 9.4],   # Patient 2 - elevated WBC, glucose
    [6.8,  5.1, 14.2, 215, 4.8],   # Patient 3 - normal
    [15.3, 3.8, 10.1, 420, 11.7],  # Patient 4 - infection indicators
    [9.4,  4.7, 13.0, 290, 6.2],   # Patient 5 - slightly elevated
])

# Extract specific data
print(f"Patient 2 full panel: {labs[1, :]}")
print(f"All WBC readings:     {labs[:, 0]}")
print(f"Patient 3-5 glucose:  {labs[2:5, 4]}")

# First two patients, first three markers
print(f"\nSubmatrix (P1-P2, WBC/RBC/Hb):\n{labs[0:2, 0:3]}")

# Column stats - mean per marker
col_means = labs.mean(axis=0)
print(f"\nMean per marker:")
for marker, mean in zip(markers, col_means):
    print(f"  {marker:<8} {mean:.2f}")

Patient 2 full panel: [ 12.1   4.2  11.8 380.    9.4]
All WBC readings:     [ 8.2 12.1  6.8 15.3  9.4]
Patient 3-5 glucose:  [ 4.8 11.7  6.2]

Submatrix (P1-P2, WBC/RBC/Hb):
[[ 8.2  4.9 13.5]
 [12.1  4.2 11.8]]

Mean per marker:
  WBC      10.36
  RBC      4.54
  Hb       12.52
  Plt      311.00
  Glucose  7.44


## 5. Views vs copies - a trap that causes real bugs

Slicing a NumPy array returns a **view** - a window into the same memory. Modifying the view modifies the original. This is different from Python lists and catches everyone once.

In [12]:
original = np.array([1, 2, 3, 4, 5, 6])
print(f"Original: {original}")

# Slicing returns a VIEW - not a copy
view = original[1:4]
print(f"View [1:4]: {view}")

view[0] = 999       # modify the view
print(f"\nAfter view[0] = 999:")
print(f"  view:     {view}")
print(f"  original: {original}")    # CHANGED - same memory!

# Confirm they share memory
print(f"\nShared memory? {np.shares_memory(original, view)}")

Original: [1 2 3 4 5 6]
View [1:4]: [2 3 4]

After view[0] = 999:
  view:     [999   3   4]
  original: [  1 999   3   4   5   6]

Shared memory? True


In [ ]:
original = np.array([10, 20, 30, 40, 50])

#solution: use .copy() when you want independence
view    = original[1:4]          # shares memory
copy    = original[1:4].copy()   # independent

view[0] = 999
copy[0] = 888

print(f"original after modifying view and copy:")
print(f"  original: {original}")   # view change visible, copy change not
print(f"  view:     {view}")
print(f"  copy:     {copy}")
print(f"\nview shares memory with original: {np.shares_memory(original, view)}")
print(f"copy shares memory with original: {np.shares_memory(original, copy)}")

# Rule: if you're about to modify a slice and need the original intact ---> .copy()
# This is CRITICAL in Pandas: df_clean = df.copy() before any inplace modifications

original after modifying view and copy:
  original: [ 10 999  30  40  50]
  view:     [999  30  40]
  copy:     [888  30  40]

view shares memory with original: True
copy shares memory with original: False


## 6. Boolean masking - preview

This deserves its own section in Class 9, but you need the concept now to make the practical useful. Boolean masking lets you filter array values using a condition, without any for loop.

In [14]:
readings = np.array([37.2, 38.9, 36.5, 39.4, 37.8, 36.9, 40.1, 38.2])

# A condition returns a boolean array
fever_mask = readings > 37.5
print(f"Readings:   {readings}")
print(f"Mask >37.5: {fever_mask}")

# Use the mask to index - pulls out only True positions
fever_values = readings[fever_mask]
print(f"Fever readings: {fever_values}")

# Count and percentage
n_fever = fever_mask.sum()    # True counts as 1
pct     = n_fever / len(readings) * 100
print(f"Fever patients: {n_fever}/{len(readings)} ({pct:.0f}%)")

# One-liner (condition applied directly in brackets)
print(f"\nAbove 39.0: {readings[readings > 39.0]}")
print(f"Normal range (36-37.5): {readings[(readings >= 36) & (readings <= 37.5)]}")

Readings:   [37.2 38.9 36.5 39.4 37.8 36.9 40.1 38.2]
Mask >37.5: [False  True False  True  True False  True  True]
Fever readings: [38.9 39.4 37.8 40.1 38.2]
Fever patients: 5/8 (62%)

Above 39.0: [39.4 40.1]
Normal range (36-37.5): [37.2 36.5 36.9]


## Practical

### Practical 1 - The dramatic speed reveal
**Context:** Your lab processes 10 million gene expression values. How long does that take with a list? With NumPy? The gap changes how you think about what's computationally feasible.

*predict: how many times faster is NumPy?*

In [15]:
import numpy as np, time

# Compute: normalise each value (subtract mean, divide by std) - classic preprocessing
N = 5_000_000
np.random.seed(42)
data = list(np.random.normal(0, 1, N))    # Python list

# Python loop
t0 = time.time()
mean_val = sum(data) / len(data)
std_val  = (sum((x - mean_val)**2 for x in data) / len(data)) ** 0.5
norm_loop = [(x - mean_val) / std_val for x in data]
t_loop = time.time() - t0

# NumPy
arr = np.array(data)
t0 = time.time()
norm_np = (arr - arr.mean()) / arr.std()
t_np = time.time() - t0

print(f"Task: z-score normalise {N:,} values")
print(f"  Python loop: {t_loop:.3f}s")
print(f"  NumPy:       {t_np:.4f}s")
print(f"  Speedup:     {t_loop/t_np:.0f}x")
print()
print(f"Results match: {np.allclose(norm_loop[:5], norm_np[:5])}")
print(f"First 5 values: {norm_np[:5].round(4)}")

Task: z-score normalise 5,000,000 values
  Python loop: 1.506s
  NumPy:       0.1131s
  Speedup:     13x

Results match: True
First 5 values: [ 0.4968 -0.1381  0.6478  1.523  -0.2339]


### Practical 2 - The view trap in a data cleaning pipeline
**Context:** An analyst extracts the 'baseline' readings from a dataset to normalise against, then accidentally corrupts the original. A common, hard-to-spot bug.

In [16]:
# Patient glucose readings over 5 days
glucose = np.array([5.1, 5.4, 9.8, 5.2, 10.3, 5.6, 11.1, 5.3])
print(f"Original readings: {glucose}")

# Analyst: "I'll grab the baseline (non-spike) readings"
baseline = glucose[glucose < 7.0]    # boolean mask - this IS a copy
print(f"Baseline readings: {baseline}")

# But slice-based extraction is a VIEW
first_half = glucose[:4]              # VIEW
first_half[0] = 0.0                  # meant to mark a missing value

print(f"\nAfter zeroing first_half[0]:")
print(f"  glucose:    {glucose}")     # modified!
print(f"  first_half: {first_half}")

# Lesson: boolean mask indexing returns a COPY; slice indexing returns a VIEW
print(f"\nShares memory:")
print(f"  glucose vs first_half: {np.shares_memory(glucose, first_half)}")
mask_result = glucose[glucose < 7.0]
print(f"  glucose vs bool mask:  {np.shares_memory(glucose, mask_result)}")

Original readings: [ 5.1  5.4  9.8  5.2 10.3  5.6 11.1  5.3]
Baseline readings: [5.1 5.4 5.2 5.6 5.3]

After zeroing first_half[0]:
  glucose:    [ 0.   5.4  9.8  5.2 10.3  5.6 11.1  5.3]
  first_half: [0.  5.4 9.8 5.2]

Shares memory:
  glucose vs first_half: True
  glucose vs bool mask:  False


### Practical 3 - dtype matters in calculations
**Context:** A student stores exam scores as int8 to save memory. Then computes the class total. What goes wrong?

In [17]:
# Scores out of 100 - looks fine as int8 (range: -128 to 127)
scores_i8  = np.array([85, 92, 78, 95, 88], dtype=np.int8)
scores_i64 = np.array([85, 92, 78, 95, 88], dtype=np.int64)

print(f"int8  total: {scores_i8.sum()}")    # correct for 5 scores
print(f"int64 total: {scores_i64.sum()}")

# Now with 3 students scoring 100 each - total = 300
big_scores = np.array([100, 100, 100], dtype=np.int8)
print(f"\nThree 100s as int8:  {big_scores.sum()}")   # 44! Overflow!
print(f"Three 100s as int64: {np.array([100,100,100]).sum()}")

# Why? int8 max is 127. 300 % 256 = 44
print(f"\n300 mod 256 = {300 % 256}  (wraps around)")
print(f"int8 max: {np.iinfo(np.int8).max}")
print(f"int8 min: {np.iinfo(np.int8).min}")

# Rule: use int8/float32 only when you've confirmed your values fit the range
# and you explicitly need the memory saving (e.g., storing 1B genomic positions)

int8  total: 438
int64 total: 438

Three 100s as int8:  300
Three 100s as int64: 300

300 mod 256 = 44  (wraps around)
int8 max: 127
int8 min: -128


### Practical 4 - Reshaping without losing data (**`Must`**)
**Context:** A sensor array delivers data as a flat stream of 12 values. You need to reshape it into a 3×4 matrix for analysis. Show what reshape does and what breaks it.

In [18]:
flat = np.arange(12)
print(f"Flat array:  {flat}")

# Reshape to 3×4
m3x4 = flat.reshape(3, 4)
print(f"\n3x4 matrix:\n{m3x4}")

# -1 means 'figure it out'
m2xN = flat.reshape(2, -1)    # 2 rows, NumPy computes 6 cols
print(f"\n2x? matrix (2,-1):\n{m2xN}")

# reshape returns a VIEW
m3x4[0, 0] = 999
print(f"\nAfter modifying m3x4[0,0]:")
print(f"  flat:   {flat}")          # changed!
print(f"  m3x4:\n{m3x4}")

# What breaks reshape: total elements must match
try:
    bad = flat.reshape(3, 3)
except ValueError as e:
    print(f"\nCan't reshape 12 elements into 3x3: {e}")

Flat array:  [ 0  1  2  3  4  5  6  7  8  9 10 11]

3x4 matrix:
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

2x? matrix (2,-1):
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]

After modifying m3x4[0,0]:
  flat:   [999   1   2   3   4   5   6   7   8   9  10  11]
  m3x4:
[[999   1   2   3]
 [  4   5   6   7]
 [  8   9  10  11]]

Can't reshape 12 elements into 3x3: cannot reshape array of size 12 into shape (3,3)


## 7. Practical - Array Slicing Lab
**Difficulty: Low–Medium**

In [19]:
# A 4×5 temperature matrix (4 patients, 5 daily readings)
temps = np.array([
    [36.5, 37.2, 38.1, 37.9, 37.4],
    [38.9, 39.4, 39.1, 38.7, 38.2],
    [36.8, 36.9, 37.0, 36.7, 36.8],
    [37.5, 38.8, 39.6, 39.2, 38.5],
])
print(f"Shape: {temps.shape}  dtype: {temps.dtype}")
print(f"Matrix:\n{temps}")

Shape: (4, 5)  dtype: float64
Matrix:
[[36.5 37.2 38.1 37.9 37.4]
 [38.9 39.4 39.1 38.7 38.2]
 [36.8 36.9 37.  36.7 36.8]
 [37.5 38.8 39.6 39.2 38.5]]


In [20]:
temps = np.array([
    [36.5, 37.2, 38.1, 37.9, 37.4],
    [38.9, 39.4, 39.1, 38.7, 38.2],
    [36.8, 36.9, 37.0, 36.7, 36.8],
    [37.5, 38.8, 39.6, 39.2, 38.5],
])

# Extract specific subsets
print("=== Extraction ===")
print(f"Patient 2 (row 1):     {temps[1, :]}")
print(f"Day 3 (col 2):         {temps[:, 2]}")
print(f"Patients 2-3 days 2-4: \n{temps[1:3, 1:4]}")
print(f"Last 2 patients:       \n{temps[-2:, :]}")
print(f"Every other reading:   \n{temps[:, ::2]}")

=== Extraction ===
Patient 2 (row 1):     [38.9 39.4 39.1 38.7 38.2]
Day 3 (col 2):         [38.1 39.1 37.  39.6]
Patients 2-3 days 2-4: 
[[39.4 39.1 38.7]
 [36.9 37.  36.7]]
Last 2 patients:       
[[36.8 36.9 37.  36.7 36.8]
 [37.5 38.8 39.6 39.2 38.5]]
Every other reading:   
[[36.5 38.1 37.4]
 [38.9 39.1 38.2]
 [36.8 37.  36.8]
 [37.5 39.6 38.5]]


In [21]:
temps = np.array([
    [36.5, 37.2, 38.1, 37.9, 37.4],
    [38.9, 39.4, 39.1, 38.7, 38.2],
    [36.8, 36.9, 37.0, 36.7, 36.8],
    [37.5, 38.8, 39.6, 39.2, 38.5],
])

# Boolean masks
fever_mask = temps > 37.5
print(f"Fever mask:\n{fever_mask}")
print(f"\nAll fever readings: {temps[fever_mask]}")
print(f"Fever count: {fever_mask.sum()} / {temps.size} readings")

# Per-patient: which patients had any fever reading?
any_fever = fever_mask.any(axis=1)   # True if ANY reading > 37.5 in that row
print(f"\nPatients with any fever: {np.where(any_fever)[0] + 1}")

# Per-patient: peak temperature
peak_temps = temps.max(axis=1)
print(f"Peak temps per patient: {peak_temps}")

Fever mask:
[[False False  True  True False]
 [ True  True  True  True  True]
 [False False False False False]
 [False  True  True  True  True]]

All fever readings: [38.1 37.9 38.9 39.4 39.1 38.7 38.2 38.8 39.6 39.2 38.5]
Fever count: 11 / 20 readings

Patients with any fever: [1 2 4]
Peak temps per patient: [38.1 39.4 37.  39.6]


In [22]:
# Speed comparison: sum of 1M elements
N = 1_000_000
data_list  = list(range(N))
data_array = np.arange(N, dtype=np.float64)

t0 = time.time()
sum_loop = sum(x**2 for x in data_list)
t_loop = time.time() - t0

t0 = time.time()
sum_np = (data_array**2).sum()
t_np = time.time() - t0

print(f"Sum of squares for {N:,} elements:")
print(f"  Python generator: {t_loop*1000:.2f} ms   result: {sum_loop:,}")
print(f"  NumPy:            {t_np*1000:.3f} ms   result: {int(sum_np):,}")
print(f"  Speedup: {t_loop/t_np:.0f}x")

Sum of squares for 1,000,000 elements:
  Python generator: 86.04 ms   result: 333,332,833,333,500,000
  NumPy:            7.655 ms   result: 333,332,833,333,500,032
  Speedup: 11x


## Summary

| Concept | Key point |
|---|---|
| Why NumPy | Contiguous memory + C execution = 20–100× faster than Python loops |
| `np.arange(s,e,step)` | Like `range()` but returns an array; step can be float |
| `np.linspace(s,e,n)` | N evenly-spaced values, both endpoints inclusive |
| `.shape` | Tuple of dimensions - check this first when debugging |
| `.dtype` | Data type - wrong dtype causes overflow or precision loss |
| `.nbytes` | Total memory - check before loading large datasets |
| Slicing = view | `a[1:4]` shares memory - use `.copy()` to get independence |
| Boolean mask | `a[a > 5]` returns elements satisfying the condition - O(n), no loop |
| `np.random.randn(r, c)` | Fills a matrix with a Standard Normal Distribution (mean=0, std=1) for ML model inputs |
| `arr.astype(dtype)` | Casts an array explicitly to a new target data type, allocating independent memory |

**Before Saturday:** What does `arr.reshape(2, -1)` do if `arr` has 15 elements? What would happen with 13 elements?